In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
# وارد کردن شبکه عصبی برای استفاده از ساختار متفاوت و Epoch
from sklearn.neural_network import MLPRegressor 
import joblib
import warnings

warnings.filterwarnings('ignore')

# ۱. بارگذاری و فیلتر داده‌ها (دقیقاً مطابق ساختار قبلی شما)
df = pd.read_csv("survey_results_public.csv") 
df = df[["Country", "EdLevel", "YearsCodePro", "Employment", "ConvertedCompYearly"]].rename({"ConvertedCompYearly": "Salary"}, axis=1).dropna()
df = df[df["Employment"] == "Employed, full-time"].drop("Employment", axis=1)

country_map = df.Country.value_counts()
df = df[df['Country'].map(country_map) >= 199]
df = df[(df["Salary"] <= 300000) & (df["Salary"] >= 10000)]

def clean_experience(x):
    if x == 'More than 50 years': return 50.0
    if x == 'Less than 1 year': return 0.5
    return float(x)

def clean_education(x):
    if 'Bachelor’s degree' in x: return 'Bachelor’s degree'
    if 'Master’s degree' in x: return 'Master’s degree'
    if 'Professional degree' in x or 'Other doctoral' in x: return 'Post grad'
    return 'Less than a Bachelors'

df['YearsCodePro'] = df['YearsCodePro'].apply(clean_experience)
df['EdLevel'] = df['EdLevel'].apply(clean_education)

X = df[['Country', 'EdLevel', 'YearsCodePro']]
y = np.log1p(df["Salary"]) # تبدیل لگاریتمی متغیر هدف برای پایداری شبکه عصبی

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=85)

# ۲. خط لوله پیش‌پدازش استاندارد برای لایه‌های ورودی شبکه عصبی
preprocessor = ColumnTransformer(
    transformers=[
        ('Num', StandardScaler(), ['YearsCodePro']),
        ('Category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Country', 'EdLevel'])
    ],
    remainder='drop' 
)

# ۳. تعریف معماری شبکه عصبی (ساختار متفاوت با اپوک مشخص)
mlp_model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32), # ساختار متفاوت: ۳ لایه پنهان با ۱۲۸، ۶۴ و ۳۲ نورون
    activation='relu',               # تابع فعال‌سازی غیرخطی ReLU
    solver='adam',                   # بهینه‌ساز پیشرفته آدام
    max_iter=100,                    # تعداد اپوک‌ها (Epochs = 100)
    batch_size=64,                   # اندازه دسته‌ها در هر گام اپوک
    learning_rate_init=0.005,        # نرخ یادگیری اولیه
    random_state=85,
    verbose=True                     # نمایش روند کاهش خطا در هر اپوک (مورد پسند اساتید)
)

# ایجاد خط لوله نهایی
nn_pipeline = Pipeline(steps=[
    ('transformer', preprocessor),
    ('model', mlp_model)
])

# آموزش مدل (در خروجی این بخش روند تغییرات اپوک‌ها را مشاهده خواهید کرد)
print("شروع آموزش شبکه عصبی با ساختار لایه‌ای تعمیم‌یافته...")
nn_pipeline.fit(X_train, y_train)

# ارزیابی مدل بر اساس واحد واقعی دلار
y_pred_log = nn_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

print("="*50)
print(f"🎯 Neural Network R2 Score: {r2_score(y_test_actual, y_pred):.4f}")
print("="*50)

# ذخیره مدل هماهنگ با وب اپلیکیشن
joblib.dump(nn_pipeline, 'salary_predictor_model.pkl')
print("✅ مدل شبکه عصبی با نام 'salary_predictor_model.pkl' ذخیره شد.")

شروع آموزش شبکه عصبی با ساختار لایه‌ای تعمیم‌یافته...
Iteration 1, loss = 1.39830239
Iteration 2, loss = 0.09872364
Iteration 3, loss = 0.09962188
Iteration 4, loss = 0.10164295
Iteration 5, loss = 0.10205076
Iteration 6, loss = 0.10256081
Iteration 7, loss = 0.10424618
Iteration 8, loss = 0.10139372
Iteration 9, loss = 0.10257845
Iteration 10, loss = 0.10414866
Iteration 11, loss = 0.10086401
Iteration 12, loss = 0.10078302
Iteration 13, loss = 0.10103315
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
🎯 Neural Network R2 Score: 0.4863
✅ مدل شبکه عصبی با نام 'salary_predictor_model.pkl' ذخیره شد.
